# HU-05 — Análisis de Valores Atípicos (Outlier Analysis)

> **PredictMain · CRISP-DM Fase: Preparación de Datos**

Este notebook es el documento ejecutable de revisión de dominio para el análisis de outliers de los sensores industriales del dataset AI4I 2020.  
Permite al validador del dominio reproducir las detecciones, revisar cada decisión y confirmar o cuestionar las justificaciones técnicas.

---

**Entradas:**  `data/processed/ai4i2020_clean.csv`  
**Salidas:**   `reports/outlier_report.md`, `data/processed/ai4i2020_outlier_analyzed.csv`

**Principio fundamental:** Los outliers **NO se eliminan automáticamente**. Cada decisión (KEEP / CAP / EXCLUDE) está justificada con criterios del dominio físico de la maquinaria.

## 1. Configuración del entorno

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Asegura que ml_service/src sea importable desde el notebook
ML_SERVICE = Path().resolve().parent  # ml_service/
if str(ML_SERVICE) not in sys.path:
    sys.path.insert(0, str(ML_SERVICE))

print(f"numpy version : {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"ml_service root: {ML_SERVICE}")

## 2. Carga del dataset limpio (HU-04 output)

In [ ]:
from src.infrastructure.paths import paths

df = pd.read_csv(paths.clean_csv_path)
print(f"Shape: {df.shape}")
print(f"Nulos totales: {df.isnull().sum().sum()}")
df.head(3)

## 3. Variables de sensores analizadas

Solo las 5 variables continuas operativas son candidatas para análisis de outliers.  
Los identificadores (`UDI`, `Product ID`) y las etiquetas binarias (`Machine failure`, modos de falla) **no aplican**.

In [ ]:
SENSOR_COLS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
df[SENSOR_COLS].describe().round(2)

## 4. Detección con IQR (1.5×) — `numpy.percentile`

Referencia: [numpy.percentile — NumPy v2 docs](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html)

La regla IQR define el rango intercuartílico `IQR = Q3 - Q1` y las vallas:
- **Fence low**  = Q1 − 1.5 × IQR  
- **Fence high** = Q3 + 1.5 × IQR

Toda observación fuera de esas vallas es un **candidato a outlier** estadístico.

In [ ]:
iqr_results = []

for col in SENSOR_COLS:
    arr = df[col].to_numpy(dtype=float)
    # np.percentile(a, q) — q en escala [0, 100]
    q1  = float(np.percentile(arr, 25, method="linear"))
    q3  = float(np.percentile(arr, 75, method="linear"))
    iqr = q3 - q1
    fence_lo = q1 - 1.5 * iqr
    fence_hi = q3 + 1.5 * iqr
    n_out = int(((df[col] < fence_lo) | (df[col] > fence_hi)).sum())

    iqr_results.append({
        "Variable": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "Fence Low": fence_lo, "Fence High": fence_hi, "IQR Outliers": n_out
    })

pd.DataFrame(iqr_results).set_index("Variable").round(2)

## 5. Detección con Percentiles P1/P99 — `numpy.percentile`

Complementa el IQR con un enfoque basado en la distribución empírica del propio dataset.

In [ ]:
pct_results = []

for col in SENSOR_COLS:
    arr = df[col].to_numpy(dtype=float)
    p1  = float(np.percentile(arr, 1,  method="linear"))   # P1
    p99 = float(np.percentile(arr, 99, method="linear"))   # P99
    n_out = int(((df[col] < p1) | (df[col] > p99)).sum())

    pct_results.append({
        "Variable": col, "P1": p1, "P99": p99, "P1/P99 Outliers": n_out
    })

pd.DataFrame(pct_results).set_index("Variable").round(2)

## 6. Límites físicos UCI — verificación de rango operativo

Los límites operativos de ingeniería del dataset AI4I 2020 (fuente: ficha técnica UCI) son la referencia definitiva para distinguir entre:

| Tipo | Criterio |
|---|---|
| **Error de medición** | Fuera del rango físico UCI |
| **Extremo válido** | Dentro del rango UCI pero estadísticamente extremo |
| **Señal de falla** | Estadísticamente extremo y correlacionado con `Machine failure = 1` |

In [ ]:
UCI_RANGES = {
    "Air temperature [K]":     (295.3, 304.5),
    "Process temperature [K]": (305.7, 313.8),
    "Rotational speed [rpm]":  (1168.0, 2886.0),
    "Torque [Nm]":             (3.8, 76.6),
    "Tool wear [min]":         (0.0, 253.0),
}

physical_check = []
for col, (lo, hi) in UCI_RANGES.items():
    n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
    physical_check.append({"Variable": col, "UCI Min": lo, "UCI Max": hi,
                            "Fuera de rango físico": n_out,
                            "Veredicto": "✅ Ninguno" if n_out == 0 else f"⚠️ {n_out} registros"})

pd.DataFrame(physical_check).set_index("Variable")

## 7. Correlación outliers IQR ↔ Machine failure

Antes de decidir qué hacer con los outliers estadísticos, verificamos si coinciden con eventos de falla real.  
Esto distingue **señales de falla** (KEEP obligatorio) de **errores de medición** (candidatos a CAP o EXCLUDE).

In [ ]:
corr_results = []

for col in SENSOR_COLS:
    arr = df[col].to_numpy(dtype=float)
    q1  = float(np.percentile(arr, 25, method="linear"))
    q3  = float(np.percentile(arr, 75, method="linear"))
    iqr = q3 - q1
    fence_lo = q1 - 1.5 * iqr
    fence_hi = q3 + 1.5 * iqr

    mask_out = (df[col] < fence_lo) | (df[col] > fence_hi)
    n_outliers = mask_out.sum()
    n_failure_in_outliers = df.loc[mask_out, "Machine failure"].sum()
    pct = (n_failure_in_outliers / n_outliers * 100) if n_outliers > 0 else 0

    corr_results.append({
        "Variable": col,
        "IQR Outliers": int(n_outliers),
        "Con Machine failure=1": int(n_failure_in_outliers),
        "% Falla en outliers": f"{pct:.1f}%"
    })

pd.DataFrame(corr_results).set_index("Variable")

## 8. Tabla de decisiones — Regla → Tipo → Decisión

Consolidación de todos los análisis anteriores en la tabla de decisión final.

In [ ]:
from src.domain.outlier_rules import OUTLIER_RULES

decision_table = pd.DataFrame([{
    "Variable":        r.column,
    "Método":          r.detection_method,
    "Tipo de atípico": r.outlier_type,
    "Decisión":        r.decision,
    "Rango UCI":       f"[{r.uci_physical_min}, {r.uci_physical_max}]",
    "Justificación":   r.rationale[:80] + "..."
} for r in OUTLIER_RULES])

decision_table.set_index("Variable")

## 9. Ejecución del pipeline completo

Reproduce los entregables oficiales de HU-05: `outlier_report.md` y `ai4i2020_outlier_analyzed.csv`.

In [ ]:
from src.application.outlier_analysis import OutlierAnalysisService

service = OutlierAnalysisService()
result = service.run(
    csv_path=paths.clean_csv_path,
    output_csv_path=paths.outlier_csv_path,
    report_path=paths.outlier_report_path,
)

print(f"Registros entrada  : {result.rows_input:,}")
print(f"Outliers detectados: {result.total_detected:,}")
print(f"Valores modificados: {result.total_modified:,}")
print(f"Filas excluidas    : {result.total_excluded:,}")
print(f"Registros salida   : {result.rows_output:,}")

## 10. Resumen por columna

In [ ]:
summary = pd.DataFrame([{
    "Variable":    s.column,
    "IQR outliers": s.iqr_outliers_count,
    "P1/P99 outliers": s.percentile_outliers_count,
    "Detectados (unión)": s.records_detected,
    "Modificados": s.records_modified,
    "Excluidos":   s.records_excluded,
    "Decisión":    s.decision,
} for s in result.column_stats])

summary.set_index("Variable")

## 11. Conclusión

| Hallazgo | Resultado |
|---|---|
| Variables analizadas | 5 sensores continuos |
| Total outliers detectados (IQR + P1/P99) | **1,170** |
| Fuera del rango físico UCI | **0** |
| Decisión aplicada | **KEEP en todos los casos** |
| Valores modificados | **0** |
| Filas excluidas | **0** |

**Conclusión del dominio:** Todos los valores estadísticamente atípicos están dentro del rango físico documentado por UCI. Los outliers de RPM y Torque son señales de precursores de falla (PWF, OSF) con alto valor predictivo para el modelo. Eliminarlos introduciría sesgo y reduciría la capacidad de detección temprana.

---
*Notebook generado para revisión del validador de dominio — HU-05 PredictMain*